# Notebook 5 — Stage 3: Fine-Tuning Qwen2.5-VL with LoRA
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook fine-tunes `Qwen/Qwen2.5-VL-7B-Instruct` on our Arabic handwriting dataset using **4-bit quantization + LoRA**.

**Dataset.** Reference transcriptions for the handwriting images were generated with **gpt-5.4-mini** and then split into train / eval JSONL files. Each sample is an OpenAI-style chat with a base64-encoded image and a JSON `{"transcription": "..."}` target.

> ⚠️ **A GPU with ≥24 GB VRAM is required.** Run this on the AWS g5.xlarge instance (NVIDIA A10G) or a Colab A100.

**Input:**
- `data/train/train.jsonl` — 1,120 samples (full train set)
- `data/eval/eval.jsonl` — 280 samples (full eval set)

**Output:** `models/lora_adapter/run-2/` — saved LoRA adapter (~380 MB). Each training run writes to its own `run-N/` subfolder; bump `RUN_NAME` in the config cell before re-training.

**Expected training time:** ~1 h on g5.xlarge (≈1,680 steps at ~30 steps/min). Run 01 used the 80-sample sample set and finished in ~2 min — the full set scales roughly 14×.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install "transformers>=4.52.0" peft==0.14.0 "accelerate>=1.3.0" "bitsandbytes>=0.46.1" torchvision
!{sys.executable} -m pip install qwen-vl-utils

## 3. Set Project Root

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/home/ubuntu/nlp_project')

assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'

print(f'Project root: {PROJECT_ROOT}')

## 4. Configuration

> ⚠️ `MIN_PIXELS`, `MAX_PIXELS`, and `DPI` **must match** what was used in the data transformation step (Notebook 4).

In [ ]:
from pathlib import Path

# --- Model ---
BASE_MODEL     = 'Qwen/Qwen2.5-VL-7B-Instruct'
MAX_SEQ_LENGTH = 8192  # CRITICAL: Do NOT set below 8192!

# --- LoRA ---
LORA_R       = 32    # Rank (max for 24 GB GPU with this model)
LORA_ALPHA   = 64    # Scaling factor (2x rank)
LORA_DROPOUT = 0.05  # Regularization
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',       # Attention
    'gate_proj', 'up_proj', 'down_proj',            # Feed-forward
]

# --- Training ---
NUM_EPOCHS             = 3
BATCH_SIZE             = 1
GRADIENT_ACCUMULATION  = 2    # Effective batch size = 2
LEARNING_RATE          = 2e-5
LR_SCHEDULER           = 'cosine'
WARMUP_RATIO           = 0.1

# --- Image processing (MUST match inference!) ---
MIN_PIXELS = 4   * 28 * 28   # 3,136
MAX_PIXELS = 128 * 28 * 28   # 100,352
DPI = 100

# --- Quantization ---
LOAD_IN_4BIT  = True
QUANT_TYPE    = 'nf4'
DOUBLE_QUANT  = True

# --- Paths ---
RUN_NAME   = 'run-2'
TRAIN_DATA = Path(PROJECT_ROOT) / 'data' / 'train' / 'train.jsonl'
EVAL_DATA  = Path(PROJECT_ROOT) / 'data' / 'eval'  / 'eval.jsonl'
OUTPUT_DIR = Path(PROJECT_ROOT) / 'models' / 'lora_adapter' / RUN_NAME
LOG_DIR    = Path(PROJECT_ROOT) / 'logs' / RUN_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'Train data: {TRAIN_DATA}')
print(f'Eval data:  {EVAL_DATA}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Log dir:    {LOG_DIR}')

## 5. Load Model with 4-bit Quantization

In [ ]:
import torch
from huggingface_hub import snapshot_download
from huggingface_hub.utils import LocalEntryNotFoundError
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

HF_CACHE_DIR = None  # None = default ~/.cache/huggingface/hub/

# Verify disk cache — download only if missing
print(f'Verifying cache for: {BASE_MODEL}')
try:
    snapshot_path = snapshot_download(
        repo_id=BASE_MODEL, local_files_only=True, cache_dir=HF_CACHE_DIR,
    )
    print(f'  Cache OK — {snapshot_path}')
except LocalEntryNotFoundError:
    print(f'  Cache MISS — downloading {BASE_MODEL} (~16 GB)...')
    snapshot_path = snapshot_download(repo_id=BASE_MODEL, cache_dir=HF_CACHE_DIR)
    print(f'  Download complete.')

# Load weights into GPU memory from verified local path
print('Loading model into GPU memory...')
quantization_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=QUANT_TYPE,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=DOUBLE_QUANT,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    snapshot_path,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa',
    local_files_only=True,
)

processor = AutoProcessor.from_pretrained(
    snapshot_path,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=True,
)

print('Model and processor ready.')

## 6. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model  # type: ignore
from peft.utils.other import prepare_model_for_kbit_training  # type: ignore

print('Applying LoRA...')

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Build the Dataset

Each JSONL row holds a chat-format `messages` list. The user turn carries the page image as a base64 `data:image/png;…` URL inside an OpenAI-style `image_url` content item; the assistant turn is the gpt-5.4-mini-generated reference transcription as a JSON string.

`extract_images()` walks the messages, base64-decodes those URLs into PIL images, and feeds them to the processor as `images=...`. Without this, the vision tower receives empty slots and the adapter learns nothing visual — this was the root cause of Run 01's degenerate outputs.

In [ ]:
import json
import base64
from io import BytesIO
from PIL import Image
from torch.utils.data import Dataset


def extract_images(messages):
    """Decode base64 data URLs from OpenAI-style `image_url` content into PIL Images."""
    images = []
    for msg in messages:
        content = msg.get('content')
        if not isinstance(content, list):
            continue
        for item in content:
            t = item.get('type')
            if t == 'image_url':
                url = item['image_url']
                if isinstance(url, dict):
                    url = url.get('url', '')
                if url.startswith('data:'):
                    b64 = url.split(',', 1)[1]
                    images.append(Image.open(BytesIO(base64.b64decode(b64))).convert('RGB'))
                elif url:
                    images.append(Image.open(url).convert('RGB'))
            elif t == 'image':
                src = item.get('image')
                if isinstance(src, Image.Image):
                    images.append(src)
                elif isinstance(src, str) and src.startswith('data:'):
                    b64 = src.split(',', 1)[1]
                    images.append(Image.open(BytesIO(base64.b64decode(b64))).convert('RGB'))
                elif isinstance(src, str):
                    images.append(Image.open(src).convert('RGB'))
    return images


class ArabicHTRDataset(Dataset):
    def __init__(self, data_path, processor, max_length):
        self.processor = processor
        self.max_length = max_length
        self.samples = []
        with open(data_path) as f:
            for line in f:
                self.samples.append(json.loads(line))
        print(f'Loaded {len(self.samples)} samples from {data_path}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        messages = sample['messages']

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

        # Decode base64 images so the vision tower actually sees them.
        # Without images=, the model trains on empty vision slots.
        image_inputs = extract_images(messages)

        # No padding here — the data collator pads each batch to its longest
        # sample. Truncation still caps any oversized sample at max_length.
        inputs = self.processor(
            text=text,
            images=image_inputs if image_inputs else None,
            return_tensors='pt',
            max_length=self.max_length,
            truncation=True,
        )
        inputs['labels'] = inputs['input_ids'].clone()
        return {k: v.squeeze(0) for k, v in inputs.items()}


print('Loading datasets...')
train_dataset = ArabicHTRDataset(TRAIN_DATA, processor, MAX_SEQ_LENGTH)
eval_dataset  = ArabicHTRDataset(EVAL_DATA,  processor, MAX_SEQ_LENGTH)

print(f'Training samples: {len(train_dataset)}')
print(f'Eval samples:     {len(eval_dataset)}')

if len(train_dataset) == 0:
    raise ValueError('NO TRAINING SAMPLES! MAX_SEQ_LENGTH is probably too low.')

## 8. Train

In [ ]:
import os
import time
from datetime import datetime
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

# Dynamic padding: pad each batch to its longest sample (not to MAX_SEQ_LENGTH).
# Pad positions in `labels` are set to -100 so they're ignored by the loss.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=processor.tokenizer,
    padding=True,
    return_tensors='pt',
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    gradient_checkpointing=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=10,
    logging_dir=str(LOG_DIR),
    report_to='none',
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

start_time = time.time()
start_dt   = datetime.now()
print(f'Training started:  {start_dt.strftime("%Y-%m-%d %H:%M:%S")}')

trainer.train()

end_dt   = datetime.now()
elapsed  = time.time() - start_time
hours, remainder = divmod(int(elapsed), 3600)
minutes, seconds = divmod(remainder, 60)
print(f'Training ended:    {end_dt.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Total time:        {hours:02d}h {minutes:02d}m {seconds:02d}s')

## 9. Save the LoRA Adapter

In [ ]:
print(f'Saving LoRA adapter to {OUTPUT_DIR}...')
model.save_pretrained(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
print('Training complete!')

# Show adapter size
import subprocess
result = subprocess.run(['du', '-sh', str(OUTPUT_DIR)], capture_output=True, text=True)
print(f'Adapter size: {result.stdout.strip()}')

## Troubleshooting

| Problem | Cause | Fix |
|---------|-------|-----|
| CUDA out of memory | LoRA rank too high or images too large | Reduce `LORA_R` to 16, reduce `MAX_PIXELS` |
| Training completed with 0 steps | `MAX_SEQ_LENGTH` too low | Set to 8192 or higher |
| `Killed` / OOM | Batch size too large | Keep `BATCH_SIZE=1`, increase `GRADIENT_ACCUMULATION` |
| Blurry images | DPI too low | Use `DPI=100` in transform step |

## Next Step

LoRA adapter saved to `models/lora_adapter/run-2/`. Continue to:

**Notebook 6 → `06_evaluation.ipynb`** — Evaluate the fine-tuned model (CER, WER, exact match) against the gpt-5.4-mini reference transcriptions.